# Fake News Detection & Retrieval: Milestone 1 - EDA & Preprocessing
**Course:** SPPU Information Retrieval (IR) Mini Project
**Dataset:** WELFake (Zenodo record 4561253)


## 1. Setup & Environment


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from ml.config import RAW, PROCESSED, LABEL_MAP, MAX_CHARS, SEED
from ml.preprocess import clean_text, preprocess, extract_stylistic_features

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Arial'


## 2. Load Raw WELFake Dataset & Inspection


In [ ]:
print(f"Loading data from: {RAW}")
df = pd.read_csv(RAW)
print(f"Dataset shape: {df.shape}")
df.head(5)


## 3. Data Cleaning: Missing Values & Duplicates
We impute null titles and texts, construct `doc = title + ' ' + text`, and drop duplicate documents.


In [ ]:
df.columns = [c.strip().lower() for c in df.columns]
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["doc"] = (df["title"] + " " + df["text"]).str.strip().str[:MAX_CHARS]
initial_count = len(df)
df = df[df["doc"].str.strip() != ""]
df = df.drop_duplicates(subset=["doc"]).reset_index(drop=True)
print(f"Removed {initial_count - len(df)} duplicate/empty rows. Cleaned size: {len(df)}")


## 4. Class Distribution & Verification of Label Mapping
WELFake dataset convention: `0 = Fake`, `1 = Real`.
Let's verify by checking samples from both classes.


In [ ]:
print("Class distribution:")
print(df["label"].value_counts())

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="label", palette=["#b42318", "#067647"], ax=ax)
ax.set_xticklabels(["0: Fake", "1: Real"])
ax.set_title("Class Balance in WELFake Dataset")
plt.show()


### Manual Verification of Label Mapping


In [ ]:
print("--- SAMPLE CLASS 0 (Fake) ---")
print(df[df["label"] == 0]["doc"].iloc[0][:300])

print("\n--- SAMPLE CLASS 1 (Real) ---")
print(df[df["label"] == 1]["doc"].iloc[0][:300])


## 5. Document Length Analysis


In [ ]:
df["word_count"] = df["doc"].apply(lambda d: len(d.split()))

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["word_count"], bins=50, kde=True, color="#0a0a0a", ax=ax)
ax.set_title("Distribution of Article Word Counts (Truncated at 3000 chars)")
ax.set_xlabel("Word Count")
plt.show()


## 6. Preprocessing Pipeline
Applies lowercase, URL removal, HTML stripping, non-alphabetic filtering, stopword removal, and WordNet lemmatization.


In [ ]:
sample_text = df["doc"].iloc[0]
print("Original:")
print(sample_text[:200])

print("\nCleaned & Lemmatized:")
print(preprocess(sample_text[:200]))


## 7. Saving Cleaned Dataset for Modeling


In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Preprocessing full corpus...")
df["clean"] = df["doc"].apply(preprocess)
df = df[df["clean"].str.strip() != ""].reset_index(drop=True)

df.to_parquet(PROCESSED / "cleaned_corpus.parquet")
print(f"Cleaned corpus saved to {PROCESSED / 'cleaned_corpus.parquet'} ({len(df)} documents).")
